# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YosraHilmi/Fly-Rank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)



## 1. Scoring followed by Ranking (supervised)
why?

Scoring (Primary Stage): Rather than treating this as a binary classification problem (e.g., "Needs Optimization" vs. "Does Not Need Optimization"), we will use a continuous numerical metric—an Opportunity Score.
Ranking (Output Stage): Once every page receives a score, we order them sequentially. Ranking converts isolated continuous scores into an actionable, prioritized queue for the content and SEO teams, allowing them to systematically focus resources on pages with the highest estimated unclaimed clicks.

## 2. proxy: (CTR Gap Opportunity)

What I would predict:
For each page, the model predicts its Expected CTR: the click-through rate that is typical for comparable pages (similar average position, search intent, content type, demand and competition). The deliverable of the project is the Opportunity Score, which is calculated from that prediction:

Opportunity Score = Impressions × max(0, Expected CTR − Actual CTR)

Where the label comes from:
The opportunity itself is a defined rule, not an observed outcome. The data has no record of what happened after a page was optimized, so there is no ground truth for "true opportunity". That is why my target is a proxy: the score estimates unclaimed clicks to prioritize pages for human review. It does not forecast clicks gained, and "expected" means typical for comparable pages, not the best a page could achieve.
The only observed label in the project is each page's historical CTR (clicks ÷ impressions), and it is used only to train the Expected CTR model.

Output: one row per page.




## 3. Success metric

Metric: Mean Absolute Error (MAE) between the predicted Expected CTR and the actual CTR, measured on pages the model has not seen during training.

Good means: a lower MAE than a simple baseline that predicts CTR from average position alone.

Note: the ranked list is the output of the project, not its metric. Since there is no ground truth for "true opportunity", the ranking itself will be checked by manual review of the top pages.

## 4. The unit of analysis, as a real dataframe

One row = one page (content item), identified by `content_id`. Each row summarizes that page's search activity over a trailing 90-day window. The dataset has 30,000 rows and 44 columns, and `content_id` is unique, so no page appears twice.

Note: `ctr` is already a percentage (0.76 means 0.76%), and `avg_position` is an average over all queries that showed the page, not a single rank.

In [8]:
import pandas as pd

url = "https://raw.githubusercontent.com/YosraHilmi/Fly-Rank-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 1. Print dataframe shape
print(f"DataFrame shape: {df.shape}")

# 2. Check if content_id is unique and print the count
is_unique = df['content_id'].is_unique
unique_count = df['content_id'].nunique()
print(f"Is content_id unique? {is_unique} ({unique_count} unique IDs)")

# 3. Build a view called unit_view with only the requested columns
requested_cols = [
    'content_id', 'impressions_90d', 'clicks_90d', 'ctr',
    'avg_position', 'position_tier', 'main_intent', 'content_type'
]

existing_cols = []
for col in requested_cols:
    if col in df.columns:
        existing_cols.append(col)
    else:
        print(f"Skipped: {col} (does not exist in dataset)")

unit_view = df[existing_cols]

# 4. Display the head of the unit_view
display(unit_view.head(10))

DataFrame shape: (30000, 44)
Is content_id unique? True (30000 unique IDs)


,content_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier,main_intent,content_type
0,content_304f48230142,3803,29,0.76,10.6,striking,transactional,keyword article
1,content_a1fb4e703a9e,15320,7,0.05,20.3,page_3_5,informational,keyword article
2,content_9aa793d4d895,12581,11,0.09,36.5,page_3_5,informational,keyword article
3,content_331d6c4de07b,11751,58,0.49,6.2,page_1,commercial,keyword article
4,content_d99b7a2d90ca,19140,24,0.13,44.0,page_3_5,informational,keyword article
5,content_d4084a4bc775,3970,1,0.03,8.5,page_1,transactional,keyword article
6,content_9a34b442b552,20,0,0.00,7.0,page_1,informational,keyword article
7,content_a63219c6e95a,1724,1,0.06,21.2,page_3_5,commercial,keyword article
8,content_5e6c160719bc,32574,29,0.09,46.0,page_3_5,informational,keyword article
9,content_c27558df2b0c,1240,2,0.16,4.9,page_1,informational,keyword article


## 5. Why ML beats a fixed rule here

A fixed rule would predict CTR from average position alone.

But pages in the same position range show very different CTRs, so position by itself does not explain CTR. Other factors, such as search intent, content type, and keyword demand and competition, also play a role, and they interact with each other, which is hard to capture in a simple if-statement. A machine-learning model can combine these factors to estimate a more suitable Expected CTR for each page. This is an expectation, not yet a result: it will be tested by comparing the model against a position-only baseline on held-out pages.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.